# bottleneck-latent-projection — ex2: tied-weight encode-decode round trip and identity for orthonormal W

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bottleneck-latent-projection`. Running the final beacon cell reports progress against the `Generative: Bottleneck latent projection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Bottleneck latent projection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bottleneck-latent-projection`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bottleneck-latent-projection"
DD_SUBTOPIC = "Generative: Bottleneck latent projection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Tied-weight decoder — quick refresher

A symmetric autoencoder bottleneck can share its weight matrix with the matching decoder layer:
```python
# encode: (B, input_dim) -> (B, latent_dim)
z = flat @ W.T + b_enc                # W shape (latent_dim, input_dim)
# decode: (B, latent_dim) -> (B, input_dim)
x_hat = z @ W + b_dec                 # SAME W, no .T this time
```

**Why tied.** Halves the parameter count and biases the decoder toward the pseudoinverse of the encoder — useful regularizer for under-trained autoencoders. Hinton's original AE work used tied weights; modern VAEs usually untie.

**Round-trip identity for orthonormal W.** If `W @ W.T == I` (rows of W are orthonormal), then `(flat @ W.T) @ W = flat @ (W.T @ W)` — and when `W` is square+orthogonal, `W.T @ W == I`, so the round-trip reproduces the input exactly (ignoring biases). This is the PCA-with-tied-weights connection: PCA's encoder/decoder pair is exactly this with W as the eigenbasis matrix.

### Exercise 2 — tied-weight encode-decode round trip and identity for orthonormal W

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the tied-weight autoencoder by implementing a round-trip encode+decode using the SAME weight matrix (W.T for encode, W for decode) and verifying that an orthonormal W reconstructs the input.
> Keywords: tied-weights, autoencoder, orthonormal, round-trip
> ```

**KCs targeted:** `tied-decoder-uses-W-not-W.T`, `orthonormal-W-yields-identity-roundtrip`

Implement `ex2_tied_encode_decode(flat_batch, weight, b_enc, b_dec)`. The classic tied-weight symmetric autoencoder pass:

1. `flat_batch` shape `(B, input_dim)`. `weight` shape `(latent_dim, input_dim)` — same convention as `nn.Linear` stores it (rows = outputs). `b_enc` shape `(latent_dim,)`. `b_dec` shape `(input_dim,)`.
2. Encode: `z = flat_batch @ weight.T + b_enc`. Shape `(B, latent_dim)`. SAME formula as ex1.
3. Decode using the SAME weight (NOT a separate `W_dec`): `x_hat = z @ weight + b_dec`. Shape `(B, input_dim)`. Note: NO `.T` on the decode — `weight` is already shaped `(latent_dim, input_dim)` which is exactly what `z @ weight` needs.
4. Return `(z, x_hat)` — both tensors as a tuple.

Critical detail: ONE weight matrix, used twice (`weight.T` on the way in, `weight` on the way out). The decoder has NO independent weight parameter.

Input: `flat_batch` `(B, input_dim)`, `weight` `(latent_dim, input_dim)`, `b_enc` `(latent_dim,)`, `b_dec` `(input_dim,)`.
Output: tuple `(z, x_hat)` with shapes `(B, latent_dim)`, `(B, input_dim)`.

The visualization renders one input image, its latent code as a bar chart, and the round-trip reconstruction — when `W` is orthonormal and biases are zero, reconstruction is identical to the input.

In [ ]:
def ex2_tied_encode_decode(flat_batch: Tensor, weight: Tensor,
                           b_enc: Tensor, b_dec: Tensor):
    """Tied-weight encode-decode round trip. Return (z, x_hat)."""
    raise NotImplementedError()


def _test_ex2():
    # Shape contract.
    B, input_dim, latent_dim = 5, 8, 8   # square so orthonormal works
    flat = t.randn(B, input_dim, generator=t.Generator().manual_seed(0))
    W = t.eye(input_dim)   # identity → orthonormal trivially
    b_enc = t.zeros(latent_dim)
    b_dec = t.zeros(input_dim)
    z, x_hat = ex2_tied_encode_decode(flat, W, b_enc, b_dec)
    assert z.shape == (B, latent_dim), f'z shape {tuple(z.shape)}'
    assert x_hat.shape == (B, input_dim), f'x_hat shape {tuple(x_hat.shape)}'

    # Identity W with zero biases must reconstruct exactly.
    assert t.allclose(x_hat, flat, atol=1e-5), 'identity W round-trip must recover input'
    assert t.allclose(z, flat, atol=1e-5), 'identity W encode must equal input'

    # A proper square orthonormal W must also round-trip exactly.
    rng = t.Generator().manual_seed(7)
    rand = t.randn(input_dim, input_dim, generator=rng)
    Q, _ = t.linalg.qr(rand)   # Q is orthonormal: Q @ Q.T = I
    z2, x_hat2 = ex2_tied_encode_decode(flat, Q, t.zeros(latent_dim), t.zeros(input_dim))
    assert t.allclose(x_hat2, flat, atol=1e-4), (
        f'orthonormal W round-trip must be identity; max diff '
        f'{(x_hat2 - flat).abs().max().item():.6f}'
    )

    # Non-square: latent_dim < input_dim. Round-trip is LOSSY but z must match ex1's projection.
    latent_dim_small = 3
    W_small = t.randn(latent_dim_small, input_dim, generator=t.Generator().manual_seed(1))
    b_enc_s = t.randn(latent_dim_small, generator=t.Generator().manual_seed(2))
    b_dec_s = t.randn(input_dim, generator=t.Generator().manual_seed(3))
    z3, x_hat3 = ex2_tied_encode_decode(flat, W_small, b_enc_s, b_dec_s)
    assert z3.shape == (B, latent_dim_small)
    assert x_hat3.shape == (B, input_dim)

    # Encode value match ex1's affine.
    expected_z = flat @ W_small.T + b_enc_s
    assert t.allclose(z3, expected_z, atol=1e-5), 'encode must equal flat @ W.T + b_enc'
    # Decode value match the analytical form (no .T on decode!).
    expected_xhat = z3 @ W_small + b_dec_s
    assert t.allclose(x_hat3, expected_xhat, atol=1e-5), (
        'decode must equal z @ W + b_dec — did you put a .T on W during decode? '
        '(The tied-weight pattern uses NO .T on the way out.)'
      )

    # Catch the most common bug: decoding with `.T` on a non-square W blows up
    # the matmul. Confirm explicitly that the BUGGY path would raise.
    buggy_raised = False
    try:
        _ = z3 @ W_small.T   # shape mismatch (5,3) @ (8,3)
    except RuntimeError:
        buggy_raised = True
    assert buggy_raised, 'decoding with .T on non-square W should raise — this drill keys off that'

    # Bias contribution: zero W must produce x_hat = b_dec (broadcast over batch).
    W_zero = t.zeros(latent_dim_small, input_dim)
    b_enc_z = t.zeros(latent_dim_small)
    b_dec_fixed = t.tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
    z_z, xhat_z = ex2_tied_encode_decode(flat, W_zero, b_enc_z, b_dec_fixed)
    for b_idx in range(B):
        assert t.allclose(xhat_z[b_idx], b_dec_fixed, atol=1e-6), (
            'zero-W round-trip must produce just b_dec broadcast over batch'
        )

    # --- Visualization: one sample, its latent, and the reconstruction ---
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    sample_idx = 0
    axes[0].bar(range(input_dim), flat[sample_idx].numpy(), color='steelblue', edgecolor='black')
    axes[0].set_title('input')
    axes[1].bar(range(latent_dim), z[sample_idx].numpy(), color='seagreen', edgecolor='black')
    axes[1].set_title('latent z (identity W → z == input)')
    axes[2].bar(range(input_dim), x_hat[sample_idx].numpy(), color='coral', edgecolor='black')
    axes[2].set_title('reconstruction (orthonormal W → exact)')
    plt.suptitle('ex2 tied-weight round trip — orthonormal W reconstructs exactly')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_tied_encode_decode(flat_batch: Tensor, weight: Tensor,
                           b_enc: Tensor, b_dec: Tensor):
    z = flat_batch @ weight.T + b_enc
    x_hat = z @ weight + b_dec
    return z, x_hat
```

**The `.T` asymmetry.** `weight` is stored as `(latent_dim, input_dim)` — rows are encoder outputs. To encode `(B, input_dim) -> (B, latent_dim)` you need the transpose. To decode `(B, latent_dim) -> (B, input_dim)` you use it as stored. ONE weight, two shapes-of-use.

**Why orthonormal W reconstructs exactly.** `x_hat = (flat @ W.T) @ W = flat @ (W.T @ W)`. For an orthonormal square `W`, `W.T @ W = I`, so the round trip is the identity. For non-square `W` (latent_dim < input_dim), `W.T @ W` is a projection onto W's row space — reconstruction is lossy by the amount of variance W's rows don't capture. This is PCA, restated.

**Tied vs untied in modern code.** Modern VAEs/AEs usually UNTIE — separate `W_enc` and `W_dec` parameters. Tying halves the parameter count and adds an implicit regularizer, but limits decoder expressivity. The tie is most useful when training data is scarce.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()